# Chess Behavior Analysis: EDA

This notebook contains EDA and visualizations only. Data collection, parsing, feature engineering, statistical testing, and modeling live in `src/`.

Key interpretation rule: **the tilt proxy is a behavioral construction, not a direct observation of psychological state**.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from src.config import load_settings
from src.viz import (plot_result_by_hour, plot_break_by_tilt, plot_heatmap,
                     plot_tilt_ci, plot_correlations)


In [ ]:
settings = load_settings('../config/config.yaml')
features_path = settings.resolve_path(settings.paths.features_csv)
stats_path = settings.resolve_path(settings.paths.stats_json)
df = pd.read_csv(features_path, parse_dates=['created_at'])
stats_result = json.loads(stats_path.read_text(encoding='utf-8'))
df.shape


## Data overview

First, inspect dataset size, missing values, and basic distributions. The notebook does not perform hidden transformations; it calls reusable functions from `src/`.

In [ ]:
display(df.head())
display(df[['user_result', 'speed', 'tilt_proxy']].value_counts(dropna=False).head(20))
display(df.isna().mean().sort_values(ascending=False).head(20).to_frame('missing_share'))


## Time of day

The canonical timestamps are stored in UTC. Calendar features use `project.timezone` from the config, defaulting to UTC. The modeling layer represents time of day with cyclic sine/cosine features.

In [ ]:
plot_result_by_hour(df, settings.resolve_path(settings.paths.figures_dir), settings.project.timezone)


## Session behavior and tilt proxy

The tilt proxy marks a current game when the previous game was a loss, the return break was short, and the preceding loss streak met the configured threshold.

In [ ]:
plot_break_by_tilt(df, settings.resolve_path(settings.paths.figures_dir))
plot_heatmap(df, settings.resolve_path(settings.paths.figures_dir))
plot_tilt_ci(stats_result, settings.resolve_path(settings.paths.figures_dir))


## Correlations

These correlations are descriptive. They do not establish causation and should not be interpreted as predictive performance.

In [ ]:
plot_correlations(df, settings.resolve_path(settings.paths.figures_dir))


## Statistical result

The statistical test compares current-game loss proportions between the tilt-proxy and control groups. The reported effect size and confidence intervals should be considered together with the p-value.

In [ ]:
summary = {
    'test': stats_result['test'],
    'p_value': stats_result['p_value'],
    'odds_ratio': stats_result['odds_ratio'],
    'odds_ratio_ci': stats_result['odds_ratio_ci'],
    'risk_difference': stats_result['risk_difference'],
    'risk_difference_ci': stats_result['risk_difference_ci'],
}
pd.Series(summary)


## Interpretation guardrails

- The proxy is not a direct measure of psychological tilt.
- Associations do not establish causation.
- This is a single-player observational dataset.
- Engine-evaluation findings apply only to games with sufficient evaluation coverage.
- The configured project timezone should be kept in mind when interpreting time-of-day patterns; the default is UTC.